# ECG HDMI / Pmod A
默认演示数据，不启动 ADC。无显示器时不能验证实体画面。

In [ ]:
from pathlib import Path
import zipfile, hashlib, json
root=Path('ecg_scope_20261023')
with zipfile.ZipFile('ecg_scope_20261023.zip') as z:z.extractall(root)
m=json.loads((root/'board/scope/build_manifest.json').read_text())
for p in ['board/scope/ecg_scope.bit','board/scope/ecg_scope.hwh']:
    assert hashlib.sha256((root/p).read_bytes()).hexdigest()==m['sha256'][p]
print('SCOPE_PACKAGE_HASH_PASS')


In [ ]:
%run ecg_scope_20261023/board/verify_overlay.py --overlay ecg_scope_20261023/board/scope/ecg_scope.bit --output ecg_scope_20261023/board/scope/filter_board_output.npy

In [ ]:
from pynq import Overlay
ol=Overlay(str(root/'board/scope/ecg_scope.bit'),download=False)
io=ol.ecg_io
assert io.read(0x3c)==0x45434731
for v in range(8):
    io.write(0x14,v)
    assert io.read(0x10)&7==v
io.write(0x14,0)
print('SCOPE_AXI_UI_PASS', 'HPD',io.read(0x38))


In [ ]:
%run ecg_scope_20261023/board/run_scope.py --source demo --duration 8 --summary ecg_scope_20261023/board/scope/board_demo_summary.json

In [ ]:
f=json.loads((root/'board/scope/filter_board_output.json').read_text())
d=json.loads((root/'board/scope/board_demo_summary.json').read_text())
assert f['passed'] and d['error'] is None and d['samples_per_channel']>=3500
assert d['overruns']==0 and d['bad_frames']==0
accept={'filter':f,'demo':d,'axi_ui':'PASS','physical_hdmi':'NOT TESTED','physical_adc':'NOT TESTED'}
(root/'board/scope/board_acceptance.json').write_text(json.dumps(accept,indent=2))
print('SCOPE_BOARD_ACCEPTANCE_PASS')
print(json.dumps(accept,indent=2))
